# Music-CRS: build LightGBM training data on Kaggle

Builds the ranking dataset in chunks of sessions (planner = Ollama qwen3:8b on the GPU, retrievers, features).
Plans for the turns of a session are requested concurrently (`PLANNER_WORKERS`), because the LLM is the bottleneck
(about 12 s/query one at a time). Each chunk is a separate CSV in `/kaggle/working/data/`, so chunks from
different runs can be combined at training time:
`python -m recommender_v2.training.train_ranker --train chunk_a.csv chunk_b.csv ...`

**Before running:** Settings -> Accelerator = GPU (T4/P100), Internet = On.
If the GitHub repo is private, add a Kaggle Secret named `GITHUB_TOKEN`.

**For the long run use _Save Version -> Save & Run All (Commit)_**, not the open tab: a commit keeps running on Kaggle's
servers when you close the browser or shut your laptop, and its outputs are saved. Files in an interactive
session are lost when it stops.

In [ ]:
# ---- settings ---------------------------------------------------------------
REPO_URL      = "https://github.com/Hemanth1102/BTP3.git"
REPO_SUBDIR   = "music-crs-baselines"   # folder inside the repo that holds recommender_v2/
BRANCH        = "main"

FIRST_SESSION = 100      # sessions 0..99 already built locally; continue from here
LAST_SESSION  = 600      # exclusive (500 new sessions; raise later if time allows)
CHUNK         = 50       # sessions per output file (smaller = less lost if a run dies)
TIME_BUDGET_H = 10.0     # stop starting new chunks after this many hours (Kaggle limit is ~12h)

PLANNER_MODEL = "qwen3:8b"
PLANNER_WORKERS = 4      # concurrent planner requests; Ollama is started with the same OLLAMA_NUM_PARALLEL

In [ ]:
import os, subprocess, sys, time, shutil
T0 = time.time()

# Kaggle images ship TensorFlow/JAX. If transformers imports them, XLA pre-allocates ~9 GB of the GPU
# and there is no room left for BGE next to Ollama (CUDA out of memory). Keep them out of the way.
os.environ.update({
    "USE_TF": "0", "USE_JAX": "0", "USE_FLAX": "0",
    "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
    "TF_FORCE_GPU_ALLOW_GROWTH": "true",
    "TF_CPP_MIN_LOG_LEVEL": "3",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})

def sh(cmd, check=True, **kw):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, **kw)
    if check and r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## 1. Get the code

In [ ]:
# Option A: a Kaggle Dataset containing the project folder (no git needed)
code_dir = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "recommender_v2" in dirs:
        code_dir = root
        break

# Option B: clone from GitHub
if code_dir is None:
    url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        print("no GITHUB_TOKEN secret; cloning anonymously (public repos only)")
    if os.path.exists("/kaggle/working/repo"):
        shutil.rmtree("/kaggle/working/repo")
    sh(f"git clone --depth 1 -b {BRANCH} {url} /kaggle/working/repo")
    code_dir = os.path.join("/kaggle/working/repo", REPO_SUBDIR)

os.chdir(code_dir)
sys.path.insert(0, code_dir)
print("code dir:", code_dir)
assert os.path.isdir("recommender_v2/training"), "recommender_v2 not found"

## 2. Python dependencies

In [ ]:
sh("pip install -q bm25s lightgbm ollama datasets omegaconf psutil 2>&1 | tail -2")
import torch
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

## 3. Start Ollama and pull the planner model

In [ ]:
sh("apt-get install -y -q zstd > /dev/null 2>&1", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", check=False)
sh("which ollama")

env = dict(os.environ, OLLAMA_KEEP_ALIVE="30m", OLLAMA_NUM_PARALLEL=str(PLANNER_WORKERS))
server = subprocess.Popen(["ollama", "serve"], env=env,
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
sh(f"ollama pull {PLANNER_MODEL}")

In [ ]:
# Quick check: the model answers, and it is on the GPU (PROCESSOR column should say 100% GPU)
import ollama
r = ollama.chat(model=PLANNER_MODEL, messages=[{"role":"user","content":"Reply with the word OK."}],
                think=False, options={"num_ctx": 4096})
print(r["message"]["content"])
sh("ollama ps")
sh('nvidia-smi --query-gpu=memory.used,memory.total --format=csv')   # expect ~6 GB used, not ~14


## 4. Build chunks
Each chunk resumes safely (`--resume`) if this cell is re-run. BGE runs on the GPU (fp16); the BGE index is
built once on the first chunk and cached.

In [ ]:
os.makedirs("/kaggle/working/data", exist_ok=True)

consecutive_failures = 0

for start in range(FIRST_SESSION, LAST_SESSION, CHUNK):
    n = min(CHUNK, LAST_SESSION - start)
    out = f"/kaggle/working/data/train_s{start:05d}_{start+n-1:05d}.csv"

    if (time.time() - T0) / 3600 > TIME_BUDGET_H:
        print(f"time budget reached; stopping before session {start}")
        break

    print(f"\n=== sessions {start}..{start+n-1} -> {out} ===")
    r = sh(
        f"python -m recommender_v2.training.build_dataset --split train "
        f"--start-session {start} --num-sessions {n} --device cuda "
        f"--planner-model {PLANNER_MODEL} --planner-workers {PLANNER_WORKERS} --output {out} --resume",
        check=False,
    )
    if r.returncode:
        consecutive_failures += 1
        print("chunk failed (re-run this cell to retry)")
        if consecutive_failures >= 2:
            print("two chunks failed in a row -> stopping; fix the error above first")
            break
    else:
        consecutive_failures = 0
    print(f"elapsed {(time.time()-T0)/3600:.2f} h")

## 5. Collect outputs

In [ ]:
sh("ls -la /kaggle/working/data | head -50")
sh("du -sh /kaggle/working/data")

In [ ]:
# Optional: train on whatever has been built so far (cross-validation) to see how the gain behaves with more data.
import glob
files = sorted(f for f in glob.glob("/kaggle/working/data/train_s*.csv"))
if files:
    sh("python -m recommender_v2.training.train_ranker --train " + " ".join(files) +
       " --cv-folds 5 --cv-rounds 100", check=False)

In [ ]:
# Stop Ollama so the run ends cleanly
server.terminate()

## Download
Output files: `/kaggle/working/data/*.csv` (+ `.meta.json`, `.queries.csv`). Download the `.csv` and `.meta.json` pairs
(the `.planner.jsonl` caches are optional but let you rebuild features cheaply, e.g. if you change a feature). Put them next to
your local `train_100.csv` and train with:

    python -m recommender_v2.training.train_ranker --train train_100.csv train_s00100_00199.csv ... --cv-folds 5